# Notebook 1: Electrode Analysis and Experimental Configuration

This notebook establishes the foundational configuration for the EEG biometric identification and authentication pipeline:
1. **Epoch Cap Calculation**: Scans all available EEG files and sessions across all subjects to identify the minimum epoch count, establishing a uniform epoch cap for consistent and fair biometric evaluation.
2. **Electrode and Region Analysis**: Maps all 58 electrodes to five anatomically grounded brain regions (Frontal, Central, Temporal, Parietal, Occipital).
3. **Model Selection and Theoretical Justification**: Formulates exactly eight advanced classifiers (4 Machine Learning and 4 Deep Learning architectures) tailored to handle complex spectral patterns.
4. **Strict Spectral Feature Protocol**: Defines the 20 spectral-domain features extracted per channel, strictly excluding all statistical, temporal, Hjorth, and entropy metrics.


In [1]:
# Setup environment, random seeds, and hardware acceleration
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mne
import torch
import xgboost as xgb
import lightgbm as lgb
import sklearn

# Reproducibility seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

mne.set_log_level("ERROR")

# Hardware acceleration check
device_str = "CUDA (GPU: " + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"MNE-Python        : {mne.__version__}")
print(f"Scikit-Learn      : {sklearn.__version__}")
print(f"XGBoost Version   : {xgb.__version__}")
print(f"LightGBM Version  : {lgb.__version__}")
print(f"Compute Device    : {device_str}")


Python Version    : 3.11.9
PyTorch Version   : 2.6.0+cu124
MNE-Python        : 1.12.1
Scikit-Learn      : 1.6.0
XGBoost Version   : 3.2.0
LightGBM Version  : 4.7.0
Compute Device    : CUDA (GPU: NVIDIA GeForce RTX 3050 Laptop GPU)


## Section A: Epoch Cap Calculation

We examine all available EEG recording files across all subjects, recording sessions, and task conditions (Closed Eyes [CE] and Open Eyes [OE]).
To ensure rigorous, unbiased, and fair biometric evaluations without sample-count imbalance artifacts across recordings, we determine the minimum number of available epochs across all files and establish it as the project-wide common epoch cap.


In [2]:
# Single code cell: Examine all available EEG files and determine the global minimum epoch cap
dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
print(f"Total EEG recording files discovered: {len(fif_files)}\n")

file_records = []
for fpath in fif_files:
    fname = fpath.name
    # Extract subject, session, and task identifiers
    parts = fpath.parts
    subject = [p for p in parts if p.startswith("sub-")][0]
    session = [p for p in parts if p.startswith("ses-")][0]
    task = "CE" if "_task-CE_" in fname else ("OE" if "_task-OE_" in fname else "UNKNOWN")
    
    # Read epochs metadata without preloading raw data into RAM
    ep = mne.read_epochs(str(fpath), preload=False, verbose=False)
    n_epochs = len(ep)
    n_channels = len(ep.ch_names)
    sfreq = ep.info['sfreq']
    
    file_records.append({
        "Subject": subject,
        "Session": session,
        "Task": task,
        "Channels": n_channels,
        "Sampling_Rate_Hz": sfreq,
        "Available_Epochs": n_epochs,
        "File_Name": fname
    })

df_files = pd.DataFrame(file_records)

# Calculate global minimum epoch cap
min_epochs = int(df_files["Available_Epochs"].min())
max_epochs = int(df_files["Available_Epochs"].max())
mean_epochs = float(df_files["Available_Epochs"].mean())
min_files = df_files[df_files["Available_Epochs"] == min_epochs]

print("=" * 80)
print(f"GLOBAL COMMON EPOCH CAP ESTABLISHED: {min_epochs} EPOCHS")
print("=" * 80)
print(f"Minimum Epochs Across Dataset : {min_epochs}")
print(f"Maximum Epochs Across Dataset : {max_epochs}")
print(f"Average Epochs Across Dataset : {mean_epochs:.1f}")
print(f"Bottleneck File(s) Determining Epoch Cap ({len(min_files)} file):")
for _, r in min_files.iterrows():
    print(f"  -> Subject: {r['Subject']} | Session: {r['Session']} | Task: {r['Task']} | File: {r['File_Name']}")
print("=" * 80)

# Display complete inspection table
print("\nDataset Inventory (All 40 Recording Files):")
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)
display(df_files[["Subject", "Session", "Task", "Channels", "Sampling_Rate_Hz", "Available_Epochs"]])


Total EEG recording files discovered: 40



GLOBAL COMMON EPOCH CAP ESTABLISHED: 45 EPOCHS
Minimum Epochs Across Dataset : 45
Maximum Epochs Across Dataset : 69
Average Epochs Across Dataset : 58.4
Bottleneck File(s) Determining Epoch Cap (1 file):
  -> Subject: sub-G2003 | Session: ses-V0 | Task: CE | File: sub-G2003_ses-V0_task-CE_desc-preproc_epo.fif

Dataset Inventory (All 40 Recording Files):


,Subject,Session,Task,Channels,Sampling_Rate_Hz,Available_Epochs
0,sub-CTR001,ses-V0,CE,58,250.0,57
1,sub-CTR001,ses-V0,OE,58,250.0,60
2,sub-CTR001,ses-V1,CE,58,250.0,59
3,sub-CTR001,ses-V1,OE,58,250.0,57
4,sub-CTR001,ses-V2,CE,58,250.0,52
5,sub-CTR001,ses-V2,OE,58,250.0,53
6,sub-CTR001,ses-V4,CE,58,250.0,57
7,sub-CTR001,ses-V4,OE,58,250.0,60
8,sub-G2001,ses-V0,CE,58,250.0,63
9,sub-G2001,ses-V0,OE,58,250.0,58


## Section B: Electrode and Region Analysis

We analyze the 58 available electrodes in the standard 10-20 extended montage. 
To investigate regional brain specialization for biometric discriminability, electrodes are categorized into five classical anatomical brain regions:
- **Frontal**: Executive control, individualized frontal lobe baseline oscillations.
- **Central**: Sensorimotor rhythms and mu rhythms, highly unique resting baseline motor signatures.
- **Temporal**: Auditory, lateralized associative networks, idiosyncratic lateral spectral signatures.
- **Parietal**: Somatosensory integration, resting parietal alpha distributions.
- **Occipital**: Primary visual cortex and dominant individual alpha peak generators.
- **ALL**: Global 58-channel montage combining all brain lobes.


In [3]:
# Inspect available montage channels and categorize into brain regions
sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
all_channels = sample_ep.ch_names
print(f"Total Available Electrodes in Montage: {len(all_channels)}")
print(f"Channels: {all_channels}\n")

# Define anatomically partitioned brain regions covering all 58 electrodes
REGIONS = {
    "Frontal": [
        "Fp1", "Fpz", "Fp2",
        "AF3", "AF4",
        "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"
    ],
    "Central": [
        "FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6",
        "C5", "C3", "C1", "Cz", "C2", "C4", "C6"
    ],
    "Temporal": [
        "T7", "T8", "TP7", "TP8", "CP5", "CP6"
    ],
    "Parietal": [
        "CP3", "CP1", "CPz", "CP2", "CP4",
        "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"
    ],
    "Occipital": [
        "PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8",
        "O1", "Oz", "O2"
    ]
}

# Verify complete partition: all 58 electrodes covered with zero overlap
allocated_channels = [ch for group in REGIONS.values() for ch in group]
assert len(allocated_channels) == len(all_channels), "Electrode count mismatch!"
assert set(allocated_channels) == set(all_channels), "Missing or extra channels detected!"
assert len(allocated_channels) == len(set(allocated_channels)), "Duplicate channels across regions!"

# Region Summary Table
region_summary = []
for region_name, ch_list in REGIONS.items():
    region_summary.append({
        "Brain Region": region_name,
        "Electrode Count": len(ch_list),
        "Electrode Channels": ", ".join(ch_list),
        "Primary Oscillatory Rhythms": "Delta, Theta, Gamma" if region_name == "Frontal" else (
            "Sensorimotor Mu, Beta" if region_name == "Central" else (
            "Temporal Theta, Gamma" if region_name == "Temporal" else (
            "Parietal Alpha, Beta" if region_name == "Parietal" else "Occipital Alpha Peak"
        )))
    })

# Add ALL montage
region_summary.append({
    "Brain Region": "ALL",
    "Electrode Count": len(all_channels),
    "Electrode Channels": "All 58 Channels Combined",
    "Primary Oscillatory Rhythms": "Full Spectral Profile (1 - 50 Hz)"
})

df_regions = pd.DataFrame(region_summary)
print("=" * 100)
print("BRAIN REGION CONFIGURATION AND ELECTRODE PARTITIONS:")
print("=" * 100)
display(df_regions[["Brain Region", "Electrode Count", "Primary Oscillatory Rhythms", "Electrode Channels"]])


Total Available Electrodes in Montage: 58
Channels: ['Fp1', 'Fpz', 'Fp2', 'AF3', 'AF4', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'T7', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP7', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'TP8', 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'PO8', 'O1', 'Oz', 'O2']

BRAIN REGION CONFIGURATION AND ELECTRODE PARTITIONS:


,Brain Region,Electrode Count,Primary Oscillatory Rhythms,Electrode Channels
0,Frontal,14,"Delta, Theta, Gamma","Fp1, Fpz, Fp2, AF3, AF4, F7, F5, F3, F1, Fz, F..."
1,Central,14,"Sensorimotor Mu, Beta","FC5, FC3, FC1, FCz, FC2, FC4, FC6, C5, C3, C1,..."
2,Temporal,6,"Temporal Theta, Gamma","T7, T8, TP7, TP8, CP5, CP6"
3,Parietal,14,"Parietal Alpha, Beta","CP3, CP1, CPz, CP2, CP4, P7, P5, P3, P1, Pz, P..."
4,Occipital,10,Occipital Alpha Peak,"PO7, PO5, PO3, POz, PO4, PO6, PO8, O1, Oz, O2"
5,ALL,58,Full Spectral Profile (1 - 50 Hz),All 58 Channels Combined


## Section C: Model Selection and Justification

We select exactly **eight advanced models**: four Machine Learning (ML) models and four Deep Learning (DL) models.
Basic baseline algorithms (such as standard decision trees or nearest centroid) are avoided in favor of models with proven mathematical foundations for capturing complex, non-linear, and high-dimensional spectral dynamics.

### Overview of Selected Models
1. **ML 1: XGBoost (Extreme Gradient Boosting)**
2. **ML 2: LightGBM (Light Gradient Boosting Machine)**
3. **ML 3: Support Vector Machine with RBF Kernel (SVM-RBF)**
4. **ML 4: Extra Trees Classifier (Extremely Randomized Trees)**
5. **DL 1: Res-MLP (Deep Residual Multi-Layer Perceptron)**
6. **DL 2: 1D-CNN (Spectral-Channel 1D Convolutional Neural Network)**
7. **DL 3: BiLSTM with Self-Attention (Bidirectional Recurrent Sequence Model)**
8. **DL 4: SpecFormer (Spectral Multi-Head Self-Attention Transformer)**

Below is the comprehensive justification table detailing suitability, nonlinear handling, impostor discrimination, and computational requirements for each model.


In [4]:
# Model Selection and Rigorous Theoretical Justification
model_justifications = [
    {
        "Model Name": "XGBoost",
        "Category": "Machine Learning",
        "Biometric Suitability": "State-of-the-art gradient boosted tree ensemble that constructs additive decision trees using second-order Taylor approximations.",
        "Nonlinear Spectral Handling": "Captures intricate multi-frequency band interactions, nonlinear power ratio thresholds, and asymmetric channel dependencies.",
        "Genuine vs Impostor Discrimination": "Utilizes adaptive gradient descent with L1/L2 leaf regularization to define sharp decision boundaries between genuine user signatures and impostors.",
        "Computational Profile": "High efficiency on multi-core CPUs; scalable histogram binning; ideal for moderate-sized biometric datasets."
    },
    {
        "Model Name": "LightGBM",
        "Category": "Machine Learning",
        "Biometric Suitability": "Leaf-wise tree expansion with Gradient-based One-Side Sampling (GOSS) and Exclusive Feature Bundling (EFB).",
        "Nonlinear Spectral Handling": "Splits leaves based on maximum loss reduction, isolating subtle, subject-specific oscillatory frequency micro-bands.",
        "Genuine vs Impostor Discrimination": "Focuses gradient updates on borderline impostor samples with large gradients, minimizing False Acceptance Rates (FAR).",
        "Computational Profile": "Ultra-fast training and inference, minimal memory footprint, excellent scalability."
    },
    {
        "Model Name": "SVM (RBF Kernel)",
        "Category": "Machine Learning",
        "Biometric Suitability": "Maximizes the geometric margin separating genuine spectral feature vectors from impostor clusters in Hilbert feature space.",
        "Nonlinear Spectral Handling": "Infinite-dimensional radial basis function projection captures complex continuous non-linear spectral manifolds.",
        "Genuine vs Impostor Discrimination": "Support vectors define the critical bounding frontier of the genuine subject's feature envelope.",
        "Computational Profile": "Moderate scaling O(N^3) during training; highly efficient inference via kernel evaluations."
    },
    {
        "Model Name": "Extra Trees",
        "Category": "Machine Learning",
        "Biometric Suitability": "Extremely randomized ensemble of decision trees with randomized cut-point selection across candidate spectral features.",
        "Nonlinear Spectral Handling": "Ensemble averaging over diverse randomized partitions yields smooth, non-linear decision boundaries that suppress sensor noise.",
        "Genuine vs Impostor Discrimination": "Substantially reduces prediction variance and prevents overfitting to spurious session-specific electrode artifacts.",
        "Computational Profile": "Faster training than standard Random Forest; embarrassingly parallel across all CPU cores."
    },
    {
        "Model Name": "Res-MLP",
        "Category": "Deep Learning",
        "Biometric Suitability": "Deep feedforward neural architecture featuring residual skip connections, Layer Normalization, and Dropout.",
        "Nonlinear Spectral Handling": "Residual pathways allow deep non-linear transformations without gradient dissipation, learning complex spectral manifolds.",
        "Genuine vs Impostor Discrimination": "Cross-entropy loss optimized with AdamW directly penalizes misclassified impostors, yielding well-calibrated probabilistic scores.",
        "Computational Profile": "GPU-accelerated tensor operations (CUDA); low parameter count preventing overfitting on tabular spectral vectors."
    },
    {
        "Model Name": "1D-CNN",
        "Category": "Deep Learning",
        "Biometric Suitability": "Multi-channel 1D convolutional network with receptive fields sliding across adjacent spectral bands and channel features.",
        "Nonlinear Spectral Handling": "Exploits spatial and frequency locality (e.g. delta-theta-alpha continuity) to build translation-invariant spectral pattern detectors.",
        "Genuine vs Impostor Discrimination": "Hierarchical feature abstraction extracts invariant spatial-frequency motifs unique to the enrolled individual.",
        "Computational Profile": "Highly parallelizable on GPU; compact kernel operations enable fast forward/backward passes."
    },
    {
        "Model Name": "BiLSTM-Attention",
        "Category": "Deep Learning",
        "Biometric Suitability": "Bidirectional LSTM processing electrode-wise spectral feature tokens, coupled with a learned Self-Attention pooling mechanism.",
        "Nonlinear Spectral Handling": "Bidirectional hidden states capture contextual dependencies across channels; attention assigns dynamic weights to salient electrodes.",
        "Genuine vs Impostor Discrimination": "Attention mechanism dynamically amplifies regions with the highest inter-subject divergence and down-weights shared background activity.",
        "Computational Profile": "GPU-accelerated recurrent units; sequential processing over electrode tokens with fast convergence."
    },
    {
        "Model Name": "SpecFormer",
        "Category": "Deep Learning",
        "Biometric Suitability": "Multi-Head Self-Attention (MHSA) Transformer encoder adapted for multi-channel spectral biometrics with positional/channel embeddings.",
        "Nonlinear Spectral Handling": "Computes all-to-all pairwise attention across all spectral bands and electrode channels simultaneously.",
        "Genuine vs Impostor Discrimination": "Captures high-order global cross-regional synchrony and spectral ratios that uniquely identify genuine individuals.",
        "Computational Profile": "Self-attention matrix computation optimized on NVIDIA GPU via PyTorch CUDA kernels."
    }
]

df_justifications = pd.DataFrame(model_justifications)
print("=" * 120)
print("COMPREHENSIVE MODEL JUSTIFICATIONS (8 ADVANCED ML/DL MODELS):")
print("=" * 120)
for _, r in df_justifications.iterrows():
    print(f"[{r['Category'].upper()}] {r['Model Name']}:")
    print(f"  * Biometric Suitability : {r['Biometric Suitability']}")
    print(f"  * Nonlinear Spectral    : {r['Nonlinear Spectral Handling']}")
    print(f"  * Impostor Separation   : {r['Genuine vs Impostor Discrimination']}")
    print(f"  * Computational Profile : {r['Computational Profile']}\n")


COMPREHENSIVE MODEL JUSTIFICATIONS (8 ADVANCED ML/DL MODELS):
[MACHINE LEARNING] XGBoost:
  * Biometric Suitability : State-of-the-art gradient boosted tree ensemble that constructs additive decision trees using second-order Taylor approximations.
  * Nonlinear Spectral    : Captures intricate multi-frequency band interactions, nonlinear power ratio thresholds, and asymmetric channel dependencies.
  * Impostor Separation   : Utilizes adaptive gradient descent with L1/L2 leaf regularization to define sharp decision boundaries between genuine user signatures and impostors.
  * Computational Profile : High efficiency on multi-core CPUs; scalable histogram binning; ideal for moderate-sized biometric datasets.

[MACHINE LEARNING] LightGBM:
  * Biometric Suitability : Leaf-wise tree expansion with Gradient-based One-Side Sampling (GOSS) and Exclusive Feature Bundling (EFB).
  * Nonlinear Spectral    : Splits leaves based on maximum loss reduction, isolating subtle, subject-specific oscillato

## Section D: Strict Spectral-Domain Feature Protocol

To comply with the strict project mandate, **only spectral-domain and frequency-band power features** are utilized.
Statistical features (mean, variance, skewness, kurtosis, peak-to-peak), Hjorth parameters (activity, mobility, complexity), entropy measures (Shannon, approximate, sample, spectral entropy), zero-crossing rates, and time-domain metrics are **strictly excluded**.

### Exact Spectral Feature Set (20 Features per Electrode):
1. **Absolute Band Powers (5)**: Delta (1-4 Hz), Theta (4-8 Hz), Alpha (8-13 Hz), Beta (13-30 Hz), Gamma (30-50 Hz) computed via Welch's PSD ($V^2/Hz$) with trapezoidal integration.
2. **Total Spectral Power (1)**: Broadband power integrated over the full 1.0 - 50.0 Hz range.
3. **Relative Band Powers (5)**: Normalized band power ratios $P_{band} / P_{total}$, invariant to overall impedance fluctuations.
4. **Spectral Band Power Ratios (4)**:
   - Theta / Beta Ratio (TBR)
   - Theta / Alpha Ratio (TAR)
   - Alpha / Beta Ratio (ABR)
   - Slow-to-Fast Ratio: $(P_\delta + P_	heta) / (P_lpha + P_eta)$
5. **Spectral Distribution & Shape Metrics (5)**:
   - Spectral Centroid: Center frequency of the power distribution ($f_c = \sum f \cdot S(f) / \sum S(f)$).
   - Spectral Spread (Bandwidth): Second central moment / dispersion around the centroid.
   - Spectral Edge Frequency 50% (SEF50): Median frequency containing 50% of spectral energy.
   - Spectral Edge Frequency 95% (SEF95): Upper roll-off frequency containing 95% of spectral energy.
   - Peak / Dominant Frequency: Frequency corresponding to maximal PSD power in 1 - 50 Hz.


In [5]:
# Define and verify the strict spectral-only feature extraction function
from scipy import signal

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

FEATURE_NAMES_PER_CH = [
    "delta_abs", "theta_abs", "alpha_abs", "beta_abs", "gamma_abs",
    "total_power",
    "delta_rel", "theta_rel", "alpha_rel", "beta_rel", "gamma_rel",
    "ratio_tbr", "ratio_tar", "ratio_abr", "ratio_slow_fast",
    "spectral_centroid", "spectral_spread", "sef50", "sef95", "peak_frequency"
]

print(f"Number of spectral features per electrode channel: {len(FEATURE_NAMES_PER_CH)}")
print(f"Feature list:\n{FEATURE_NAMES_PER_CH}\n")

# Verify feature extraction on one trial of actual dataset data
test_ep = mne.read_epochs(str(fif_files[0]), preload=True, verbose=False)
raw_data = test_ep.get_data()[:min_epochs]  # shape: (45, 58, 1251)
sfreq = test_ep.info['sfreq']

# Compute Welch PSD along time axis
freqs, psd = signal.welch(raw_data, fs=sfreq, nperseg=min(256, raw_data.shape[-1]), axis=-1)
print(f"Raw data shape  : {raw_data.shape} (epochs, channels, times)")
print(f"PSD shape       : {psd.shape} (epochs, channels, frequencies)")
print(f"Frequency range : {freqs[0]:.1f} Hz to {freqs[-1]:.1f} Hz ({len(freqs)} frequency bins)")

# Verification: Confirm absence of any non-spectral features
assert len(FEATURE_NAMES_PER_CH) == 20
non_spectral_keywords = ["mean", "std", "skew", "kurtosis", "ptp", "hjorth", "entropy", "zero_crossing", "energy"]
for feat in FEATURE_NAMES_PER_CH:
    for kw in non_spectral_keywords:
        assert kw not in feat, f"Violation: non-spectral feature '{feat}' detected!"

print("\n[STRICT REQUIREMENT VERIFIED] 100% of features are strictly frequency-band and spectral-domain metrics.")
print(f"Common Epoch Cap: {min_epochs} epochs established.")
print("Notebook 1 configuration completed successfully.")


Number of spectral features per electrode channel: 20
Feature list:
['delta_abs', 'theta_abs', 'alpha_abs', 'beta_abs', 'gamma_abs', 'total_power', 'delta_rel', 'theta_rel', 'alpha_rel', 'beta_rel', 'gamma_rel', 'ratio_tbr', 'ratio_tar', 'ratio_abr', 'ratio_slow_fast', 'spectral_centroid', 'spectral_spread', 'sef50', 'sef95', 'peak_frequency']



Raw data shape  : (45, 58, 1251) (epochs, channels, times)
PSD shape       : (45, 58, 129) (epochs, channels, frequencies)
Frequency range : 0.0 Hz to 125.0 Hz (129 frequency bins)

[STRICT REQUIREMENT VERIFIED] 100% of features are strictly frequency-band and spectral-domain metrics.
Common Epoch Cap: 45 epochs established.
Notebook 1 configuration completed successfully.
